# 23-F Delivery Pipeline

Single notebook to run the full end-to-end flow:
1. La Moncloa scraping
2. PDF download
3. RTVE catalog scraping
4. Text extraction + OCR + base corpus (pipeline.py)
5. Analytical corpus build with OCR quality checks (build_corpus.py)
6. NLP vocabulary

This notebook merges the operational flow from main.py and pipeline.py into a cell-based execution workflow.

This notebook generates:

- data/metadata/documents_enriched.csv
- data/metadata/document_corpus.csv
- outputs/sprint1/manual_validation_sample.csv
- outputs/sprint1/top_words_overall.csv
- outputs/sprint1/top_words_by_ministry.csv
- outputs/sprint1/nlp_preprocess_summary.txt
- outputs/sprint1/figx_yy_zz.png

In [1]:
from pathlib import Path
import os
import sys

# Detect the repo root robustly, no matter where you run this notebook from.
CANDIDATES = [Path.cwd(), *Path.cwd().parents]
ROOT = None
for p in CANDIDATES:
    if (p / 'main.py').exists() and (p / 'pipeline.py').exists():
        ROOT = p
        break

if ROOT is None:
    raise RuntimeError('Project root not found (main.py/pipeline.py).')

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src' / 'data_etl'))

print(f'ROOT: {ROOT}')
print(f'Python: {sys.executable}')

ROOT: /home/albercc/project-23f
Python: /home/albercc/project-23f/.venv/bin/python


In [ ]:
# Main imports for the flow (main.py + pipeline.py + sprint1).
import pandas as pd

from get_data import run_scraper, run_download, show_status
from pipeline import run as run_pipeline
from src.data_etl.rtve_scraper import scrape_all
from src.data_etl.build_corpus import build_corpus
from src.sprint1.manual_validation_sample import build_sample

# Operational parameters (adjustable).
MAX_EXTRACTION_WORKERS = 4
# Keeping OCR at 1 process avoids frequent EasyOCR+GPU multiprocessing failures.
MAX_OCR_WORKERS = 2
MANUAL_SAMPLE_N = 15
MANUAL_SAMPLE_SEED = 23
TOP_K = 30
MIN_LEN = 3

print('Configuration loaded.')

Configuration loaded.


## 0) Initial status

In [3]:
show_status()


PROJECT STATUS

Downloaded PDFs: 0 (directory does not exist)
Extracted texts: 0 (directory does not exist)
Detected links: 0 (scraping has not been done)


## 1) La Moncloa scraping (main.py --scrape)

Generates data/metadata/moncloa_links.csv

In [4]:
ok_scrape = run_scraper()
if not ok_scrape:
    raise RuntimeError('La Moncloa scraping failed.')


STEP 1: LA MONCLOA SCRAPING
Accessing: https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx

Total documents found: 155

By Ministry:
 ministry
Defensa       108
Interior       28
Exteriores     19

By Category:
 category
General    155
Saved to: data/metadata/moncloa_links.csv

✓ Scraping completed: 155 PDFs detected


## 2) PDF download (main.py --download)

Downloads to data/raw/ and logs data/metadata/download_log.csv

In [5]:
ok_download = run_download()
if not ok_download:
    raise RuntimeError('PDF download failed.')


STEP 2: PDF DOWNLOAD
DOWNLOADER - 23-F Documents La Moncloa

Total PDFs to download : 155
Destination directory  : data/raw
Concurrent threads     : 3
Rate limit per thread  : 0.5s



Downloading:   0%|          | 0/155 [00:00<?, ?it/s]

Downloading: 100%|██████████| 155/155 [02:07<00:00,  1.21it/s]


SUMMARY
status
success    155

✓ 155 PDFs  (284.6 MB)

Log: data/metadata/download_log.csv

✓ Download completed: 155/155 PDFs


## 3) Scraping RTVE

Genera data/metadata/rtve_documents.csv

In [6]:
rtve_df = scrape_all()
if rtve_df.empty:
    raise RuntimeError('RTVE extraction failed or returned an empty table.')

rtve_out = ROOT / 'data' / 'metadata' / 'rtve_documents.csv'
rtve_out.parent.mkdir(parents=True, exist_ok=True)
rtve_df.to_csv(rtve_out, index=False, encoding='utf-8-sig')
print(f'RTVE documents saved: {len(rtve_df)} to {rtve_out}')

=== Scraper 23F RTVE ===


[Page 1]
  → 167 resultados
  → Página 1 de 1
  → Accumulated: 167

✅ Total documents in listing: 167

📄 Fetching summaries and OCR text (167 pages, 8 threads)...


Details:  50%|█████     | 84/167 [00:30<00:28,  2.92it/s]09:53:46  WARNING   Retrying (Retry(total=0, connect=None, read=False, redirect=None, status=None)) after connection broken by 'MustDowngradeError('The server yielded its support for HttpVersion.h3 through the Alt-Svc header while unable to do so. To remediate that issue, either disable HttpVersion.h3 or reach out to the server admin.')': /document/ocr/1859
09:53:46  WARNING   Retrying (Retry(total=0, connect=None, read=False, redirect=None, status=None)) after connection broken by 'MustDowngradeError('The server yielded its support for HttpVersion.h3 through the Alt-Svc header while unable to do so. To remediate that issue, either disable HttpVersion.h3 or reach out to the server admin.')': /document/ocr/1858
09:53:46  WARNING   Retrying (Retry(total=0, connect=None, read=False, redirect=None, status=None)) after connection broken by 'MustDowngradeError('The server yielded its support for HttpVersion.h3 through the Alt-Svc heade

RTVE documents saved: 167 to /home/albercc/project-23f/data/metadata/rtve_documents.csv


## 4) Main pipeline (pipeline.py --ocr)

Performs text extraction, OCR on scanned/image PDFs, and builds a base corpus. Set apply_ocr to False to extract only from non-scanned PDFs. This yields a smaller dataset, but it is much faster to process and analyze, with enough text quality for many use cases and for validating correct pipeline behavior without waiting for OCR, which is the most expensive stage.

In [7]:
df_final = run_pipeline(
    apply_ocr=True,
    max_extraction_workers=MAX_EXTRACTION_WORKERS,
    max_ocr_workers=MAX_OCR_WORKERS,
    force_reextract=False,
    save=True,
)

print('df_final shape:', df_final.shape)
print('Moncloa docs:', (df_final['source'] == 'Moncloa').sum())
print('RTVE docs:', (df_final['source'] == 'RTVE').sum())

09:53:50  INFO      Loaded: 155 Moncloa links, enriching with extraction results
09:53:50  INFO      Loaded: 155 Moncloa docs, 167 RTVE docs
09:53:50  INFO      === Step 1: PDF text extraction (pdfplumber) ===
09:53:50  INFO        155 PDFs to extract (4 threads)
Extracting: 100%|██████████| 155/155 [00:22<00:00,  6.87it/s]
09:54:13  INFO        Done: 155 extracted, 0 errors
09:54:15  INFO      === Step 2: OCR for scanned PDFs (EasyOCR) ===
09:54:16  WARNING     GPU/MPS OCR with multiple processes can fail. Forcing max_ocr_workers=1.
09:54:16  WARNING     Set OCR_FORCE_MULTI_GPU=1 to keep multi-process GPU OCR.
09:54:16  INFO        89 PDFs to OCR (1 processes, EasyOCR GPU/MPS)
OCR: 100%|██████████| 89/89 [15:22<00:00, 10.37s/it]
10:09:40  INFO        Done: 89 OCR'd, 0 bad extractions, 0 errors
10:09:40  INFO      === Step 3: Loading extracted texts ===
10:09:40  INFO        155/155 texts loaded (0 missing)
10:09:40  INFO      Saved: /home/albercc/project-23f/data/metadata/documents_en

df_final shape: (322, 13)
Moncloa docs: 155
RTVE docs: 167


## 5) Enriched analytical corpus (build_corpus.py)

Rewrites document_corpus.csv with OCR quality columns and canonical NLP text:
- ocr_quality_score
- flag_illegible
- analysis_text

In [8]:
build_corpus(extract_metadata=False)

df_corpus = pd.read_csv('data/metadata/document_corpus.csv')
print('document_corpus shape:', df_corpus.shape)
print('columns:', df_corpus.columns.tolist())
if 'flag_illegible' in df_corpus.columns:
    print('Moncloa illegible:', int(df_corpus[(df_corpus['source'] == 'Moncloa') & (df_corpus['flag_illegible'] == True)].shape[0]))

Loading Moncloa documents...
  155 documents loaded.
Loading RTVE documents...
  167 documents loaded.

Scanning corpus for OCR quality...
Data cleaning completed: identified 8 potentially illegible documents.

Applying simple rules to infer doc_type...

Normalizing dates and calculating precision...

Corpus saved to data/metadata/document_corpus.csv
Total documents : 322
  Moncloa (legible) : len=147
  Moncloa (illegible) : len=8
  RTVE          : 167
document_corpus shape: (322, 17)
columns: ['doc_id', 'source', 'title', 'url', 'ministry', 'category', 'filename', 'date', 'date_precision', 'doc_type', 'tags', 'extracted_text', 'extracted_text_length', 'rtve_summary', 'ocr_quality_score', 'flag_illegible', 'analysis_text']
Moncloa illegible: 8


## 6) Alber Sprint 1 - Manual validation sample

Generates outputs/sprint1/manual_validation_sample.csv

In [9]:
sample = build_sample(sample_size=MANUAL_SAMPLE_N, seed=MANUAL_SAMPLE_SEED)
out_sample = ROOT / 'outputs' / 'sprint1' / 'manual_validation_sample.csv'
out_sample.parent.mkdir(parents=True, exist_ok=True)
sample.to_csv(out_sample, index=False, encoding='utf-8')
print(f'Sample saved: {out_sample}')
print(f'Rows: {len(sample)}')
display(sample.head(10))

Sample saved: /home/albercc/project-23f/outputs/sprint1/manual_validation_sample.csv
Rows: 15


,name,url,ministry,category,filename,rel_path,char_count,is_scanned,success,manual_date,manual_doc_type,date_ok,doc_type_ok,needs_ocr,notes
2,Conversaciones telefónicas de (presuntamente) ...,https://www.lamoncloa.gob.es/consejodeministro...,Interior,General,23F_3._Conversaciones_telefonicas_unidad_milit...,Interior/General/23F_3._Conversaciones_telefon...,193,True,True,,,,,True,
4,Documento manuscrito de posible planificación ...,https://www.lamoncloa.gob.es/consejodeministro...,Interior,General,23F_5._Documento_manuscrito_planificacion_del_...,Interior/General/23F_5._Documento_manuscrito_p...,110,True,True,,,,,True,
22,Índices de subversión en las FAS. Marca: SECRE...,https://www.lamoncloa.gob.es/consejodeministro...,Interior,General,2_Indices_de_subversion_en_las_FAS_DIC_1981.pdf,Interior/General/2_Indices_de_subversion_en_la...,126,True,True,,,,,True,
27,"Notas de 1983 sobre ""Libertad de condenados po...",https://www.lamoncloa.gob.es/consejodeministro...,Interior,General,7_Notas_1983_desp.pdf,Interior/General/7_Notas_1983_desp.pdf,193,True,True,,,,,True,
31,Resumen de la actuación del Departamento de De...,https://www.lamoncloa.gob.es/consejodeministro...,Defensa,General,Documento_4_R.pdf,Defensa/General/Documento_4_R.pdf,3360,False,True,,,,,False,
54,Vista oral 2/81 del Consejo Supremo de Justici...,https://www.lamoncloa.gob.es/consejodeministro...,Defensa,General,Documento_27_R.pdf,Defensa/General/Documento_27_R.pdf,13050,False,True,,,,,False,
56,Vista oral 2/81 del Consejo Supremo de Justici...,https://www.lamoncloa.gob.es/consejodeministro...,Defensa,General,Documento_29_R.pdf,Defensa/General/Documento_29_R.pdf,126,True,True,,,,,True,
66,Vista oral 2/81 del Consejo Supremo de Justici...,https://www.lamoncloa.gob.es/consejodeministro...,Defensa,General,Documento_39_R.pdf,Defensa/General/Documento_39_R.pdf,14678,False,True,,,,,False,
70,Vista oral 2/81 del Consejo Supremo de Justici...,https://www.lamoncloa.gob.es/consejodeministro...,Defensa,General,Documento_43_R.pdf,Defensa/General/Documento_43_R.pdf,159,True,True,,,,,True,
72,Información integrada (20 de abril de 1982).,https://www.lamoncloa.gob.es/consejodeministro...,Defensa,General,Documento_45_R.pdf,Defensa/General/Documento_45_R.pdf,1992,False,True,,,,,False,


## 7) NLP vocabulary by ministry

Run the sprint script to generate:
- outputs/sprint1/top_words_overall.csv
- outputs/sprint1/top_words_by_ministry.csv
- outputs/sprint1/nlp_preprocess_summary.txt

In [10]:
import subprocess

cmd = [
    sys.executable,
    str(ROOT / 'src' / 'sprint1' / 'nlp_vocab_by_ministry.py'),
    '--top-k', str(TOP_K),
    '--min-len', str(MIN_LEN),
]
result = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('nlp_vocab_by_ministry.py failed')

Saved:
- /home/albercc/project-23f/outputs/sprint1/top_words_overall.csv
- /home/albercc/project-23f/outputs/sprint1/top_words_by_ministry.csv
- /home/albercc/project-23f/outputs/sprint1/nlp_preprocess_summary.txt

